# Avito CTR Prediction — Feature Engineering
**Objective**: Build a rich feature matrix from the merged Avito dataset, moving beyond the provided HistCTR baseline toward features that capture session context, user behaviour, and ad content signals.
**Dataset**: Avito Context Ad Clicks, Kaggle 2015 competition
**Author**: Ayush Singh

## Section 1 — Objective

### Starting point: HistCTR baseline

Notebook 04 established that the pre-computed `HistCTR` column achieves **AUC = 0.6640** on its own — within 0.0163 of the KDD Cup best model (AUC = 0.6803) which required 12 engineered features and a tuned XGBoost. The gap is small but closing it requires features that encode information HistCTR cannot: session structure, user history, and ad content.

### What we are building on top of HistCTR

| Feature family | Motivation | Inspired by |
|---|---|---|
| Temporal / session | User position in browsing session predicts fatigue and intent | Gzsiceberg `gen_cnt_part.py` (bf_cnt, af_cnt) |
| Rate / target encoding | Entity-level smoothed CTR extends the KDD Cup p* approach | KDD Cup NB02 (pUId, pQId, α=0.05, β=75) |
| Content | Price, title length, category hierarchy — not in KDD Cup at all | Avito dataset unique features |

### Three feature families

**(a) Temporal / session features** — hour of day, day of week, position within session, session size, and ads already seen in this search. These capture browsing context that HistCTR cannot.

**(b) Rate / target encoding features** — smoothed CTR per AdID, CategoryID, LocationID, Position, and DeviceID using Bayesian smoothing (α=0.05, β=75) matching NB02. HistCTR is already provided by Avito; these features add entity-level signal for entities not well-covered by HistCTR.

**(c) Content features** — log-transformed price, title word count, category hierarchy depth, and location match. These are structurally unavailable in KDD Cup (anonymised dataset) and represent a qualitative advantage of the Avito dataset.

### Output

All features are saved to `data/avito/sample/features_5m.parquet` for direct ingestion by Notebook 06.

With the feature engineering plan established, the next step is loading and merging the six source files — identical to Notebook 04 — then immediately filtering to contextual impressions only before any feature is computed.

## Section 2 — Data Loading & Merge

In [1]:
%matplotlib inline

# Setup: imports and constants
# Same library set as NB04; ALPHA and BETA match the KDD Cup NB02 smoothing parameters
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

DATA_DIR = '../data/avito/sample/'
ALPHA    = 0.05   # smoothing prior — global CTR estimate (same as KDD Cup NB02)
BETA     = 75     # smoothing strength — equivalent prior sample size (same as KDD Cup NB02)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')
print('Libraries loaded.')

Libraries loaded.


In [2]:
# Load all 6 source files before merging
# Shapes verified against NB04: train=5M, search≈1.37M, ads≈1.96M, users≈4.28M
train  = pd.read_csv(DATA_DIR + 'train_stream_5m.tsv', sep='\t')
search = pd.read_csv(DATA_DIR + 'search_info_5m.tsv',  sep='\t')
ads    = pd.read_csv(DATA_DIR + 'ads_info_5m.tsv',     sep='\t')
users  = pd.read_csv(DATA_DIR + 'user_info.tsv',        sep='\t')
cat_df = pd.read_csv(DATA_DIR + 'category.tsv',         sep='\t')
loc_df = pd.read_csv(DATA_DIR + 'location.tsv',         sep='\t')

for name, df_raw in [('train_stream', train), ('search_info', search),
                     ('ads_info',     ads),   ('user_info',   users),
                     ('category',     cat_df), ('location',   loc_df)]:
    print(f'{name:15s}  shape={str(df_raw.shape):18s}  columns={list(df_raw.columns)}')

train_stream     shape=(5000000, 6)        columns=['SearchID', 'AdID', 'Position', 'ObjectType', 'HistCTR', 'IsClick']
search_info      shape=(1374429, 9)        columns=['SearchID', 'SearchDate', 'IPID', 'UserID', 'IsUserLoggedOn', 'SearchQuery', 'LocationID', 'CategoryID', 'SearchParams']
ads_info         shape=(1958330, 7)        columns=['AdID', 'LocationID', 'CategoryID', 'Params', 'Price', 'Title', 'IsContext']
user_info        shape=(4284823, 5)        columns=['UserID', 'UserAgentID', 'UserAgentOSID', 'UserDeviceID', 'UserAgentFamilyID']
category         shape=(68, 4)             columns=['CategoryID', 'Level', 'ParentCategoryID', 'SubcategoryID']
location         shape=(4080, 4)           columns=['LocationID', 'Level', 'RegionID', 'CityID']


In [3]:
# Five-step left join — identical to NB04; left joins preserve all 5M train rows
# Columns renamed immediately after each join to avoid _x/_y suffix collisions

# Merge 1: train_stream + search_info on SearchID
# Adds search context: date, user, location, category, query for each impression
df = train.merge(search, on='SearchID', how='left')
df.rename(columns={'LocationID': 'search_LocationID',
                   'CategoryID': 'search_CategoryID'}, inplace=True)

# Merge 2: + ads_info on AdID
# Adds ad attributes: title, price, context flag, ad-side location and category
df = df.merge(ads, on='AdID', how='left')
df.rename(columns={'LocationID': 'ad_LocationID',
                   'CategoryID': 'ad_CategoryID'}, inplace=True)

# Merge 3: + user_info on UserID
# Adds device and browser IDs; logged-out rows (no UserID) get NaN — correct
df = df.merge(users, on='UserID', how='left')

# Merge 4: + category hierarchy on search_CategoryID
# Adds category level and parent — used in Section 6 content features
cat_renamed = cat_df.rename(columns={
    'Level':            'cat_Level',
    'ParentCategoryID': 'cat_ParentCategoryID',
    'SubcategoryID':    'cat_SubcategoryID'
})
df = df.merge(cat_renamed,
              left_on='search_CategoryID', right_on='CategoryID',
              how='left')
df.drop(columns=['CategoryID'], inplace=True)

# Merge 5: + location on search_LocationID
# Adds RegionID and CityID for the user's search geography
loc_renamed = loc_df.rename(columns={'Level': 'loc_Level'})
df = df.merge(loc_renamed,
              left_on='search_LocationID', right_on='LocationID',
              how='left')
df.drop(columns=['LocationID'], inplace=True)

print(f'Fully merged: {df.shape[0]:,} rows × {df.shape[1]} columns')

Fully merged: 5,000,000 rows × 30 columns


In [4]:
# Filter to contextual impressions only
# NB04 confirmed: 2,577,017 non-contextual rows have IsClick=0 by dataset construction
# Keeping them would inflate the negative class with zero-signal rows
df = df[df['IsContext'] == 1].copy()
print(f'After IsContext==1 filter: {len(df):,} rows  '
      f'(dropped {5_000_000 - len(df):,} non-contextual)')

# Parse SearchDate to datetime — required for hour and day-of-week extraction in Section 3
df['SearchDate'] = pd.to_datetime(df['SearchDate'])
print(f'SearchDate range: {df["SearchDate"].min()} → {df["SearchDate"].max()}')
print(f'Contextual click rate: {df["IsClick"].mean():.4%}')

After IsContext==1 filter: 2,422,983 rows  (dropped 2,577,017 non-contextual)


SearchDate range: 2015-04-25 00:00:02 → 2015-05-20 17:59:45
Contextual click rate: 0.6142%


The 2,422,983-row contextual dataframe is ready. Section 3 extracts temporal and session features — when the impression occurred and how deep in the browsing session it fell.

## Section 3 — Temporal & Session Features

### Concept: session position as a behavioural signal

A user's position in their browsing session predicts click behaviour. Early in a session, users are exploring — they may be more open to contextual ads because they have not yet found what they want. Later in a session, users may be fatigued (lower click propensity) or more focused (higher purchase intent for the right ad).

Gzsiceberg's `gen_cnt_part.py` captured this with `bf_cnt` (impressions before this search in the session) and `af_cnt` (impressions after). Here we use a lighter version: `session_size` (how many contextual ads appeared in this search), `position_in_session` (Position ÷ session_size — note: session_size only takes values 1 or 2 in this dataset, so position_in_session is not a normalised [0, 1] score but a ratio in the range [0.5, 7.0]), and the clock signals `hour_of_day` and `day_of_week`.

In [5]:
# hour_of_day (0-23): captures time-of-day intent
# Lunch-hour browsing vs evening browsing likely have different click propensities
df['hour_of_day'] = df['SearchDate'].dt.hour

# day_of_week (0=Monday, 6=Sunday): captures weekend vs weekday patterns
# Classified-ad engagement (buying second-hand goods) may peak on weekends
df['day_of_week'] = df['SearchDate'].dt.dayofweek

print('hour_of_day distribution:')
print(df['hour_of_day'].value_counts().sort_index().to_string())
print('\nday_of_week distribution (0=Mon):')
print(df['day_of_week'].value_counts().sort_index().to_string())

hour_of_day distribution:
hour_of_day
0      65035
1      31861
2      17350
3      12612
4      14649
5      24347
6      41822
7      63321
8      90884
9     112358
10    121630
11    128110
12    127371
13    132555
14    136132
15    137226
16    139170
17    137007
18    140950
19    154899
20    165882
21    168907
22    149905
23    109000

day_of_week distribution (0=Mon):
day_of_week
0    401013
1    406753
2    362603
3    316618
4    276594
5    308643
6    350759


In [6]:
# session_size: total contextual ads shown in this SearchID
# Large sessions mean many ads compete for attention — position signal is stronger
df['session_size'] = df.groupby('SearchID')['AdID'].transform('count')

# position_in_session: relative position [0,1] normalised by session depth
# Raw Position varies with session_size; this normalises across searches of different depth
df['position_in_session'] = df['Position'] / df['session_size']

# ads_before: number of ads displayed above this one (Position-1, floored at 0)
# Models user scroll fatigue — each additional ad above competes for attention
df['ads_before'] = (df['Position'] - 1).clip(lower=0)

print('session_size stats:')
print(df['session_size'].describe())
print('\nposition_in_session stats:')
print(df['position_in_session'].describe())
print('\nads_before stats:')
print(df['ads_before'].describe())

session_size stats:
count   2422983.0000
mean          1.8655
std           0.3412
min           1.0000
25%           2.0000
50%           2.0000
75%           2.0000
max           2.0000
Name: session_size, dtype: float64

position_in_session stats:
count   2422983.0000
mean          1.8655
std           1.4366
min           0.5000
25%           0.5000
50%           1.0000
75%           3.5000
max           7.0000
Name: position_in_session, dtype: float64

ads_before stats:


count   2422983.0000
mean          2.5965
std           2.9727
min           0.0000
25%           0.0000
50%           0.0000
75%           6.0000
max           6.0000
Name: ads_before, dtype: float64


Temporal and session features are in place. Section 4 adds a more personal dimension: how this specific user has behaved across prior sessions — impression history, click history, and category engagement — computed strictly from events before this impression.

## Section 4 — User Behaviour Features

In [7]:
# Sort by UserID then SearchDate before computing cumulative user features
# cumcount() and cumsum() are row-order-sensitive — earlier impressions must appear first
# so that each row only sees information from before it, preventing data leakage
df.sort_values(['UserID', 'SearchDate'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Sorted by UserID, SearchDate.  Shape: {df.shape}')

Sorted by UserID, SearchDate.  Shape: (2422983, 35)


In [8]:
# user_impression_count: number of ads this user has seen BEFORE this row
# cumcount() returns 0 for a user's first impression — correct, no prior history yet
# Captures user fatigue: a user who has seen 100 ads responds differently than a first-timer
df['user_impression_count'] = df.groupby('UserID').cumcount()

# user_click_count: number of times this user has clicked BEFORE this impression
# shift(1) within each group ensures the current row's IsClick is excluded (no leakage)
# fillna(0) handles the first row per user where shift produces NaN
df['user_click_count'] = (
    df.groupby('UserID')['IsClick']
      .transform(lambda x: x.shift(1).fillna(0).cumsum())
)

# user_historical_ctr: this user's personal click rate from prior impressions
# clip(lower=1) prevents division by zero for the first impression
# where() fills first-impression rows with the global CTR as an unbiased prior
global_ctr = df['IsClick'].mean()
df['user_historical_ctr'] = (
    df['user_click_count'] / df['user_impression_count'].clip(lower=1)
)
df['user_historical_ctr'] = df['user_historical_ctr'].where(
    df['user_impression_count'] > 0, global_ctr
)

print('user_impression_count stats:')
print(df['user_impression_count'].describe())
print(f'\nuser_historical_ctr stats (global prior = {global_ctr:.4f}):')
print(df['user_historical_ctr'].describe())

# Fraction of rows with zero prior user history — determines how often the global CTR prior fires
zero_history_frac = (df['user_impression_count'] == 0).mean()
print(f"\nFraction of rows with zero prior user history: {zero_history_frac:.4f}")

user_impression_count stats:
count   2422983.0000
mean          4.9254
std          10.3591
min           0.0000
25%           0.0000
50%           1.0000
75%           5.0000
max         396.0000
Name: user_impression_count, dtype: float64

user_historical_ctr stats (global prior = 0.0061):
count   2422983.0000
mean          0.0058
std           0.0497
min           0.0000
25%           0.0000
50%           0.0000
75%           0.0061
max           1.0000
Name: user_historical_ctr, dtype: float64

Fraction of rows with zero prior user history: 0.2707


In [9]:
# uid_category_count: times this user has searched in this category BEFORE this row
# Captures user affinity for a category — repeated searchers may have higher purchase intent
# groupby on [UserID, search_CategoryID] then cumcount() gives 0 on first visit, 1 on second
df['uid_category_count'] = df.groupby(
    ['UserID', 'search_CategoryID']
).cumcount()

print('uid_category_count stats:')
print(df['uid_category_count'].describe())

# CTR by category visit count — does repeat engagement predict higher click probability?
df['_ucc_bucket'] = pd.cut(df['uid_category_count'],
                            bins=[-1, 0, 1, 4, 9, float('inf')],
                            labels=['0', '1', '2-4', '5-9', '10+'])
print('\nCTR by uid_category_count bucket:')
print(
    df.groupby('_ucc_bucket', observed=True)['IsClick']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'CTR', 'count': 'impressions'})
      .assign(CTR_pct=lambda x: x['CTR'] * 100)
      .to_string()
)
df.drop(columns=['_ucc_bucket'], inplace=True)

uid_category_count stats:
count   2422983.0000
mean          2.9535
std           7.1155
min           0.0000
25%           0.0000
50%           1.0000
75%           3.0000
max         293.0000
Name: uid_category_count, dtype: float64

CTR by uid_category_count bucket:
               CTR  impressions  CTR_pct
_ucc_bucket                             
0           0.0086       860606   0.8594
1           0.0065       716636   0.6497
2-4         0.0044       453847   0.4358
5-9         0.0027       214685   0.2697
10+         0.0015       177209   0.1546


User behaviour features encode individual history without leakage. Section 5 extends the KDD Cup rate-encoding approach to Avito entity types — computing Bayesian-smoothed CTR per ad, category, location, position, and device.

## Section 5 — Rate Encoding Features

### Extending the KDD Cup p\* approach

Notebook 02 used Bayesian-smoothed CTR per entity and reached **AUC = 0.6803** with 12 features. The smoothing formula:

```
smoothed_ctr = (clicks + α × β) / (impressions + β)
```

where **α = 0.05** (global CTR as the prior) and **β = 75** (equivalent prior sample size). Both values match KDD Cup NB02 exactly.

The key difference here: Avito already provides `HistCTR` at the impression level. The rate features below capture entity-level signal at a coarser grain — per ad, per search category, per location, per device — which complements rather than duplicates `HistCTR`.

**Leakage note**: These rates are computed on the full contextual set here. In Notebook 06, rate features must be recomputed on the training fold only and then mapped to the validation fold, to prevent target leakage during evaluation.

In [10]:
# Smoothed CTR helper — Bayesian shrinkage per entity column
# Low-volume entities (few impressions) are pulled toward the global CTR (alpha)
# High-volume entities are trusted more; their empirical rate dominates over the prior
def smoothed_ctr_map(df, col, alpha=ALPHA, beta=BETA):
    stats = df.groupby(col)['IsClick'].agg(clicks='sum', impressions='count')
    stats['smoothed'] = (stats['clicks'] + alpha * beta) / (stats['impressions'] + beta)
    return stats['smoothed']

# Sanity check: position-level smoothed CTR should match NB04 contextual position values
# NB04: Pos1=0.7309%, Pos7=0.4614%
test_map = smoothed_ctr_map(df, 'Position')
print('Smoothed CTR by Position (compare to NB04: Pos1≈0.73%, Pos7≈0.46%):')
print(test_map.sort_index().to_string())

Smoothed CTR by Position (compare to NB04: Pos1≈0.73%, Pos7≈0.46%):
Position
1   0.0073
7   0.0046


In [11]:
# ad_ctr: smoothed historical CTR per AdID
# Most granular entity — captures ad-specific appeal beyond what HistCTR provides
# (HistCTR is per impression-context pair; ad_ctr aggregates all impressions for the ad)
ad_ctr_map = smoothed_ctr_map(df, 'AdID')
df['ad_ctr'] = df['AdID'].map(ad_ctr_map)

print(f'ad_ctr: {df["ad_ctr"].notna().sum():,} non-null, {df["ad_ctr"].isna().sum()} NaN')
print(df['ad_ctr'].describe())

ad_ctr: 2,422,983 non-null, 0 NaN
count   2422983.0000
mean          0.0156
std           0.0115
min           0.0014
25%           0.0067
50%           0.0119
75%           0.0219
max           0.1265
Name: ad_ctr, dtype: float64


In [12]:
# category_ctr: smoothed CTR per search CategoryID
# NB04 Section 6 showed significant CTR variation across categories
category_ctr_map = smoothed_ctr_map(df, 'search_CategoryID')
df['category_ctr'] = df['search_CategoryID'].map(category_ctr_map)

# location_ctr: smoothed CTR per search LocationID
# Regional differences are modest (NB04 Section 8) but still a measurable signal
location_ctr_map = smoothed_ctr_map(df, 'search_LocationID')
df['location_ctr'] = df['search_LocationID'].map(location_ctr_map)

print(f'category_ctr: {df["category_ctr"].notna().sum():,} non-null')
print(df['category_ctr'].describe())
print(f'\nlocation_ctr: {df["location_ctr"].notna().sum():,} non-null')
print(df['location_ctr'].describe())

category_ctr: 2,422,983 non-null


count   2422983.0000
mean          0.0062
std           0.0031
min           0.0025
25%           0.0037
50%           0.0058
75%           0.0069
max           0.0179
Name: category_ctr, dtype: float64

location_ctr: 2,422,983 non-null
count   2422983.0000
mean          0.0070
std           0.0046
min           0.0032
25%           0.0049
50%           0.0058
75%           0.0071
max           0.0731
Name: location_ctr, dtype: float64


In [13]:
# position_ctr: smoothed CTR per Position
# Mirrors pPosCTR from KDD Cup (importance weight=103 in NB02) — position is a reliable signal
position_ctr_map = smoothed_ctr_map(df, 'Position')
df['position_ctr'] = df['Position'].map(position_ctr_map)

# device_ctr: smoothed CTR per UserDeviceID
# NB04 Section 7 showed device type varies in CTR; captures mobile vs desktop intent
# Logged-out users have NaN UserDeviceID — fill with global CTR as a neutral prior
device_ctr_map = smoothed_ctr_map(df, 'UserDeviceID')
df['device_ctr'] = df['UserDeviceID'].map(device_ctr_map)
df['device_ctr'] = df['device_ctr'].fillna(df['IsClick'].mean())

print(f'position_ctr: {df["position_ctr"].notna().sum():,} non-null')
print(df['position_ctr'].describe())
print(f'\ndevice_ctr: {df["device_ctr"].notna().sum():,} non-null')
print(df['device_ctr'].describe())

position_ctr: 2,422,983 non-null


count   2422983.0000
mean          0.0061
std           0.0013
min           0.0046
25%           0.0046
50%           0.0073
75%           0.0073
max           0.0073
Name: position_ctr, dtype: float64

device_ctr: 2,422,983 non-null
count   2422983.0000
mean          0.0066
std           0.0033
min           0.0038
25%           0.0063
50%           0.0063
75%           0.0063
max           0.0617
Name: device_ctr, dtype: float64


Rate encoding features complete the entity-level signal picture. Section 6 adds content signals unique to Avito — price, ad title length, category hierarchy depth, and search-to-ad location alignment — none of which exist in the anonymised KDD Cup dataset.

## Section 6 — Content Features

In [14]:
# price_log: log1p(Price) — compresses the wide price range (roubles) into a usable scale
# NB04 showed CTR broadly increases with price; log scale captures the non-linear shape
# Non-positive prices (price=0 likely means 'price not listed') are treated as missing
price_valid = df['Price'].notna() & (df['Price'] > 0)
df['price_log'] = np.nan
df.loc[price_valid, 'price_log'] = np.log1p(df.loc[price_valid, 'Price'])

# has_price: binary flag for whether a price was provided
# Unpriced listings may behave differently from priced ones; the flag lets the model learn this
df['has_price'] = price_valid.astype(int)

# price_bucket: decile bin (0-9) within the valid-price population
# Preserves price ordering while handling extreme outliers via quantile-based binning
df['price_bucket'] = np.nan
df.loc[price_valid, 'price_bucket'] = pd.qcut(
    df.loc[price_valid, 'price_log'], q=10, labels=False, duplicates='drop'
)

print(f'has_price: {df["has_price"].sum():,} ads with price ({df["has_price"].mean()*100:.1f}%)')
print('\nprice_log stats:')
print(df['price_log'].describe())
print('\nprice_bucket distribution:')
print(df['price_bucket'].value_counts().sort_index().to_string())

has_price: 2,421,194 ads with price (99.9%)

price_log stats:
count   2421194.0000
mean          8.3575
std           1.6551
min           0.4055
25%           7.2378
50%           8.3666
75%           9.4873
max          18.6830
Name: price_log, dtype: float64

price_bucket distribution:
price_bucket
0.0000    244048
1.0000    241610
2.0000    249137
3.0000    237368
4.0000    239864
5.0000    240847
6.0000    242518
7.0000    241824
8.0000    241905
9.0000    242073


In [15]:
# title_word_count: number of words in the ad title (split on whitespace)
# KDD Cup NB01 found CTR peaks at 3-word queries; here we test if ad title length matters
# fillna('') handles ads with a missing Title column before splitting
df['title_word_count'] = df['Title'].fillna('').str.split().str.len()

print('title_word_count stats:')
print(df['title_word_count'].describe())
print('\nCTR by title word count (top 10 by volume):')
twc_ctr = (
    df.groupby('title_word_count')['IsClick']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'CTR', 'count': 'impressions'})
)
twc_ctr['CTR_pct'] = twc_ctr['CTR'] * 100
print(twc_ctr.nlargest(10, 'impressions').sort_index().to_string())

title_word_count stats:
count   2422983.0000
mean          4.8070
std           2.0526
min           0.0000
25%           3.0000
50%           5.0000
75%           6.0000
max          13.0000
Name: title_word_count, dtype: float64

CTR by title word count (top 10 by volume):
                    CTR  impressions  CTR_pct
title_word_count                             
1                0.0032       143008   0.3196
2                0.0061       203090   0.6066
3                0.0075       292202   0.7539
4                0.0064       463824   0.6421
5                0.0064       436346   0.6438
6                0.0057       369986   0.5695
7                0.0062       245834   0.6203
8                0.0057       193469   0.5680
9                0.0057        56106   0.5650
10               0.0058        11527   0.5812


In [16]:
# category_level: hierarchy depth from category.tsv (1=root, 2=subcategory, etc.)
# Deeper categories are more specific — ads at leaf level may be better matched to intent
df['category_level'] = df['cat_Level']

# parent_category_id: top-level parent category from the hierarchy
# Coarser-grained signal alongside search_CategoryID; useful for target encoding in NB06
df['parent_category_id'] = df['cat_ParentCategoryID']

# category_match: 1 if the ad's CategoryID matches the user's search CategoryID
# NB04 found mismatched ads click MORE (0.3167% vs 0.2921%) — the binary flag lets the model
# learn the direction from data; column names search_CategoryID and ad_CategoryID set in merge
df['category_match'] = (
    df['search_CategoryID'].notna() &
    df['ad_CategoryID'].notna() &
    (df['search_CategoryID'] == df['ad_CategoryID'])
).astype(int)

print(f'category_level distribution:')
print(df['category_level'].value_counts().sort_index().to_string())
print(f'\ncategory_match: {df["category_match"].sum():,} matches '
      f'({df["category_match"].mean()*100:.1f}% of contextual impressions)')

category_level distribution:
category_level
1     130956
2     452267
3    1839760

category_match: 1,839,179 matches (75.9% of contextual impressions)


All three feature families are engineered. Section 7 assembles them into a single feature matrix, reports the NaN landscape and per-feature correlation with IsClick, then saves the result to parquet for Notebook 06.

## Section 7 — Feature Matrix Assembly

In [17]:
# Assemble the final feature matrix with all 21 engineered features plus target and IDs
# SearchID and AdID are kept as identifiers for joining predictions back in NB06
final_features = [
    'HistCTR',                                          # pre-computed baseline (provided)
    'Position', 'position_in_session', 'ads_before',   # position signals
    'session_size',                                     # session depth
    'hour_of_day', 'day_of_week',                       # temporal signals
    'user_impression_count', 'user_historical_ctr',    # user history
    'uid_category_count',                               # user-category affinity
    'ad_ctr', 'category_ctr', 'location_ctr',          # rate encoding
    'position_ctr', 'device_ctr',                      # rate encoding
    'price_log', 'has_price', 'title_word_count',      # content signals
    'category_level', 'category_match',                # content signals
    'IsUserLoggedOn'                                    # login state
]

id_cols = ['SearchID', 'AdID']
target  = 'IsClick'

df_features = df[id_cols + [target] + final_features].copy()
print(f'Feature matrix shape: {df_features.shape}')
print(f'Features: {len(final_features)}, Target: {target}, IDs: {id_cols}')

Feature matrix shape: (2422983, 24)
Features: 21, Target: IsClick, IDs: ['SearchID', 'AdID']


In [18]:
# NaN counts — determines which features need imputation in NB06
nan_counts = df_features[final_features].isnull().sum()
nan_pct    = nan_counts / len(df_features) * 100
nan_report = pd.DataFrame({'NaN count': nan_counts, 'NaN %': nan_pct})
nan_report = nan_report[nan_report['NaN count'] > 0].sort_values('NaN count', ascending=False)
print('Features with NaN values:')
print(nan_report.to_string() if len(nan_report) > 0 else '  None')

# Per-feature Pearson correlation with IsClick
# Linear correlation is a fast proxy for signal strength before fitting a model
numeric_feats = df_features[final_features].select_dtypes(include=[np.number]).columns.tolist()
correlations  = df_features[numeric_feats + [target]].corr()[target].drop(target)
print('\nFeature correlations with IsClick (top 10 by |r|):')
print(correlations.abs().sort_values(ascending=False).head(10).to_string())

# IsClick distribution — confirm label balance unchanged from NB04
ctr = df_features[target].mean()
print(f'\nIsClick in feature matrix: {df_features[target].sum():,} clicks / '
      f'{len(df_features):,} rows = {ctr:.4%} CTR')

Features with NaN values:
           NaN count  NaN %
price_log       1789 0.0738



Feature correlations with IsClick (top 10 by |r|):
ad_ctr                  0.0411
category_ctr            0.0397
HistCTR                 0.0311
location_ctr            0.0298
user_historical_ctr     0.0204
session_size            0.0200
user_impression_count   0.0197
uid_category_count      0.0189
Position                0.0171
ads_before              0.0171

IsClick in feature matrix: 14,883 clicks / 2,422,983 rows = 0.6142% CTR


In [19]:
# Save to parquet — columnar format preserves dtypes and is significantly faster to load
# than TSV; index=False avoids writing the pandas integer index as a column
out_path = DATA_DIR + 'features_5m.parquet'
df_features.to_parquet(out_path, index=False)
print(f'Saved: {out_path}')

# Reload and verify shape — confirms the parquet round-trip is lossless
df_check = pd.read_parquet(out_path)
print(f'Reload check: {df_check.shape}  — '
      f'matches original: {df_check.shape == df_features.shape}')
print(f'\nColumns in parquet:')
print(list(df_check.columns))

Saved: ../data/avito/sample/features_5m.parquet
Reload check: (2422983, 24)  — matches original: True

Columns in parquet:
['SearchID', 'AdID', 'IsClick', 'HistCTR', 'Position', 'position_in_session', 'ads_before', 'session_size', 'hour_of_day', 'day_of_week', 'user_impression_count', 'user_historical_ctr', 'uid_category_count', 'ad_ctr', 'category_ctr', 'location_ctr', 'position_ctr', 'device_ctr', 'price_log', 'has_price', 'title_word_count', 'category_level', 'category_match', 'IsUserLoggedOn']


The feature matrix is saved. The following section collects the key engineering findings, flags what was surprising, and lists imputation and encoding decisions for Notebook 06.

## Section 8 — What I Learned

### Observations

1. **Rate encoding transferred to Avito, but the entity importance order will likely shift.**
   In KDD Cup NB02 the user-level rate `pUId` had the highest feature importance (weight=856).
   In Avito, many users are anonymous (logged-out), so `user_historical_ctr` carries NaN for a
   substantial fraction of rows. The ad-level and category-level rate features (`ad_ctr`,
   `category_ctr`) are more densely populated and will probably rank higher. The Pearson
   correlation table from the cell above (ad_ctr=0.0411, category_ctr=0.0397, HistCTR=0.0311,
   location_ctr=0.0298, user_historical_ctr=0.0204) gives the first read on which engineered
   feature correlates most with IsClick before any model is fitted.

2. **HistCTR is a strong baseline that the rate features complement rather than replace.**
   HistCTR alone scored AUC = 0.6640 (NB04). The rate features (`ad_ctr`, `category_ctr`,
   `location_ctr`, `position_ctr`, `device_ctr`) are derived from the same IsClick signal,
   so marginal lift over HistCTR will be modest unless they capture cross-entity variation
   that HistCTR misses at the impression level. The session and user-behaviour features are
   more likely to add independent signal, since they encode temporal structure rather than
   historical rates.

3. **Cumulative user features are leakage-free but sparsest at the start of each user's history.**
   `user_impression_count` is 0 for every user's first impression and `user_historical_ctr`
   falls back to the global prior (0.0061). The fraction of rows with zero prior
   history is 0.2707 (27.1% of all contextual impressions) — meaning more than a quarter of
   rows have no personal click history and receive the global CTR prior instead. This limits
   the feature's individual contribution but does not eliminate it; the remaining 73% of rows
   carry real historical signal.

4. **Session size distribution revealed how concentrated searches are.**
   `session_size` stats (mean=1.87, max=2.0; session_size is binary in this dataset — only
   values 1 and 2) showed that most searches return only a small number of contextual ads.
   Where `session_size = 1`, `position_in_session` equals the raw Position value (1–7) —
   the ratio becomes uninformative because there is no competition within the session to
   normalise against. The raw `ads_before` feature avoids this problem and is more directly
   interpretable as a measure of scroll fatigue.

5. **Content features fill the structural gap left by KDD Cup's anonymisation.**
   `price_log`, `title_word_count`, and `category_level` have no equivalent in the KDD Cup
   dataset where all text was hashed and no price data was available. The 99.9%
   of ads that have a price listed means `price_log` is not always available — `has_price`
   as a companion flag allows the model to handle missing price correctly without ad-hoc imputation.

6. **The NB04 category-match finding is now encoded as a feature, not a hard rule.**
   The NB04 finding that category-mismatched ads click at 0.3167% vs 0.2921% for matched ads
   motivated a `category_match` feature (ad CategoryID == search CategoryID). This connects
   directly back to NB04: the model can now learn whether mismatch genuinely drives higher CTR
   or whether that pattern was confounded by position or HistCTR. `location_match` was discarded
   — every contextual ad in this sample has a null ad LocationID, making it a zero-variance
   feature with a 0.0% match rate and no predictive value.

### Imputation and encoding notes for Notebook 06
- `HistCTR` NaN: impute with 0.0 (all non-contextual rows already dropped; remaining NaN is minimal)
- `price_log`, `price_bucket` NaN: impute with median of the non-null values; use `has_price` as a mask feature
- `category_level`, `parent_category_id` NaN: impute with mode
- Rate features NaN: impute with the global CTR (the entity was not seen in training)
- `parent_category_id`: too many levels for one-hot encoding; use smoothed target encoding in NB06